In [0]:
CREATE DATABASE IF NOT EXISTS silver_db;
SHOW DATABASES;


databaseName
bronze_db
default
information_schema
silver_db


In [0]:
USE silver_db;
SELECT current_database();

current_schema()
silver_db


### Pipeline Silver incrémental
#### 1.Créer la table si elle n'existe pas

In [0]:
-- Créer la table si elle n'existe pas
--Supprimer la table processed_trips si elle existe
--Créer la table processed_trips 
-- Avec les conditions ci-dessous:
-- passenger_count > 0
-- trip_distance > 0
-- total_amont > 0
-- dropoff_datetime > pickup_datetime
-- payment_type = 1
CREATE TABLE IF NOT EXISTS processed_trips
USING DELTA
AS
SELECT * 
FROM bronze_db.raw_trips
WHERE passenger_count > 0
  AND trip_distance > 0
  AND total_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime
  AND tip_amount >= 0
  AND payment_type = 1;
  

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) FROM processed_trips;

count(1)
9617774


In [0]:
SELECT DISTINCT file_name
FROM processed_trips 
ORDER BY file_name;

file_name
yellow_tripdata_2024-01.parquet
yellow_tripdata_2024-02.parquet
yellow_tripdata_2024-03.parquet
yellow_tripdata_2024-04.parquet


#### 2. Charger seulement les nouvelles données de la table Bronze

In [0]:
with new_data AS(
    SELECT 
        * 
    FROM bronze_db.raw_trips b
    WHERE b.ingestion_timetamp > (
        SELECT MAX(ingestion_timetamp) 
        FROM processed_trips
        )
    AND b.passenger_count > 0
    AND b.trip_distance > 0
    AND b.total_amount > 0
    AND b.tpep_pickup_datetime < tpep_dropoff_datetime 
    AND b.tip_amount >= 0
    AND b.payment_type = 1
    )

    MERGE INTO processed_trips s
    USING new_data n
    ON s.file_name = n.file_name
    AND s.tpep_pickup_datetime = n.tpep_pickup_datetime
    AND s.tpep_dropoff_datetime = n.tpep_dropoff_datetime
    AND s.ingestion_timetamp = n.ingestion_timetamp
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2674590,0,0,2674590


In [0]:
SELECT COUNT(*)
FROM processed_trips

count(1)
12292364


In [0]:
 SELECT *
 FROM processed_trips
 LIMIT 10;

vendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,ingestion_timetamp,file_name
2,2024-03-01T00:33:45.000Z,2024-03-01T00:39:34.000Z,1,0.82,1,N,164,162,1,7.9,1.0,0.5,1.29,0.0,1.0,14.19,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:08:23.000Z,2024-03-01T00:17:53.000Z,1,2.15,1,N,161,141,1,12.1,1.0,0.5,5.13,0.0,1.0,22.23,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:24:58.000Z,2024-03-01T00:30:31.000Z,1,1.1,1,N,236,237,1,8.6,1.0,0.5,2.04,0.0,1.0,15.64,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:49:40.000Z,2024-03-01T01:01:25.000Z,1,2.78,1,N,161,114,1,14.9,1.0,0.5,2.0,0.0,1.0,21.9,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
1,2024-03-01T00:39:16.000Z,2024-03-01T00:48:12.000Z,1,1.2,1,N,164,246,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:42:29.000Z,2024-03-01T00:57:36.000Z,1,3.22,1,N,246,79,1,17.7,1.0,0.5,4.54,0.0,1.0,27.24,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:59:29.000Z,2024-03-01T01:11:20.000Z,1,5.34,1,N,79,263,1,22.6,1.0,0.5,5.52,0.0,1.0,33.12,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
1,2024-03-01T00:34:52.000Z,2024-03-01T00:48:12.000Z,1,2.1,1,N,186,141,1,12.1,3.5,0.5,2.0,0.0,1.0,19.1,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
1,2024-03-01T00:55:33.000Z,2024-03-01T01:07:05.000Z,1,2.4,1,N,237,234,1,12.8,3.5,0.5,4.45,0.0,1.0,22.25,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
2,2024-03-01T00:09:52.000Z,2024-03-01T00:23:48.000Z,1,1.99,1,N,148,256,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.0,2026-11-05T10:06:31.573Z,yellow_tripdata_2024-03.parquet
